[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/41_capstone_projects/90_exercises.ipynb)

# Bài tập — Chương 41: Capstone (L0 → L7)

Các bài dùng dữ liệu thô `credit_applications_raw.csv` của bài 02. Hàm làm sạch và đặc trưng tham chiếu được cho sẵn ở ô dưới (bài 5 yêu cầu bạn tự viết lại một phần).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/41_capstone_projects

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import CREDIT_MONTHLY_RATE, make_credit_applications
from ml_course.utils import repo_path

raw = pd.read_csv(repo_path("data", "sample", "credit_applications_raw.csv"))
REGION_MAP = {"tp.hcm": "TP.HCM", "ho chi minh": "TP.HCM", "hcm": "TP.HCM", "tp hcm": "TP.HCM", "hà nội": "Hà Nội", "ha noi": "Hà Nội",
              "hn": "Hà Nội", "đà nẵng": "Đà Nẵng", "da nang": "Đà Nẵng", "khác": "Khác"}
NUMERIC = ["age", "log_income", "loan_amount_vnd", "term_months", "n_dependents", "credit_history_months",
           "late_payments_12m", "dti", "loan_to_income", "income_missing"]
CATEGORICAL = ["employment_type", "purpose", "region", "has_collateral"]


def clean_applications(raw):
    df = raw.drop_duplicates().drop_duplicates("application_id").copy()
    dates = df.application_date.astype(str)
    df["application_date"] = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce").fillna(pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce"))
    df["loan_amount_vnd"] = df.loan_amount_vnd.astype(str).str.replace(".", "", regex=False).astype(float)
    income = df.monthly_income_vnd.astype(float)
    df["monthly_income_vnd"] = np.where(income < 1000, income * 1e6, income)
    df["age"] = df.age.where(df.age.between(18, 100))
    emp = df.employment_type.astype("string").str.strip().str.lower().str.replace("[- ]", "_", regex=True)
    df["employment_type"] = emp.replace({"hop_dong": "contract"}).fillna("unknown").astype(str)
    df["region"] = df.region.astype(str).str.strip().str.lower().map(REGION_MAP)
    return df.sort_values(["application_date", "application_id"]).reset_index(drop=True)


def add_features(df):
    out = df.copy()
    annuity = CREDIT_MONTHLY_RATE / (1 - (1 + CREDIT_MONTHLY_RATE) ** -out.term_months)
    out["dti"] = out.loan_amount_vnd * annuity / out.monthly_income_vnd
    out["log_income"] = np.log(out.monthly_income_vnd)
    out["loan_to_income"] = out.loan_amount_vnd / out.monthly_income_vnd
    out["income_missing"] = out.monthly_income_vnd.isna().astype(int)
    return out


def make_model(numeric=NUMERIC):
    columns = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
                                 ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
    return make_pipeline(columns, LogisticRegression(max_iter=2000))


apps = add_features(clean_applications(raw))

## L0 — Nhớ khái niệm

**Câu 1.** (a) Vì sao capstone cần baseline **nghiệp vụ** chứ không chỉ baseline "dự đoán tỉ lệ nền"? (b) Nhãn "chưa trưởng thành" là gì, xử lý thế nào? (c) Làm sao nhận ra một cột là thông tin **sau quyết định**? (d) Vì sao quyết định theo lợi nhuận kỳ vọng cần xác suất **được hiệu chỉnh**? (e) Nhãn sớm (proxy) dùng để làm gì và rủi ro của nó? (f) "Hợp đồng dữ liệu" là gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Đọc tóm tắt capstone sau và chỉ ra ít nhất 5 vấn đề: *"Chúng tôi dự đoán vỡ nợ bằng XGBoost với 500 cây, dùng toàn bộ 16 cột (kể cả `early_dpd30_3m`), chia ngẫu nhiên 80/20, điền nhãn thiếu bằng 0. AUC test 0.93, cao hơn hồi quy logistic 0.01. Ngưỡng 0.5 cho kết quả accuracy 92%. Mô hình sẵn sàng triển khai."*

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Khoản vay 100 triệu, biên lãi ròng 5%/năm, tỉ lệ mất khi vỡ nợ 70%. (a) Ngưỡng hoà vốn `p_star_36` cho kỳ hạn 36 tháng; (b) lợi nhuận kỳ vọng `ev_24` (triệu đồng) của khoản 24 tháng với $p = 0.1$; (c) nhãn sớm có TPR = 0.55, FPR = 0.04; tỉ lệ nhãn sớm tháng này 0.12 → tỉ lệ vỡ nợ ước lượng `rate_from_proxy`.

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
p_star_36 = None
ev_24 = None
rate_from_proxy = None
### END CODE HERE ###
print(round(p_star_36, 4), round(ev_24, 2), round(rate_from_proxy, 4))

In [ ]:
check_close(p_star_36, 0.1765, tol=1e-3, message="ngưỡng hoà vốn 36 tháng = g/(g+ℓ)")
check_close(ev_24, 2.0, message="lợi nhuận kỳ vọng = 0.9 × 10 − 0.1 × 70 (triệu)")
check_close(rate_from_proxy, 0.1569, tol=1e-3, message="suy ngược tỉ lệ vỡ nợ từ nhãn sớm")

## L3 — Code

**Bài 4.** Viết lại hai bước làm sạch khó nhất: `parse_dates(series)` (chuỗi `yyyy-mm-dd` hoặc `dd/mm/yyyy` → `datetime64`, không đọc được → NaT) và `parse_amount(series)` (chuỗi `"120000000"` hoặc `"120.000.000"` → float).

In [ ]:
def parse_dates(series: pd.Series) -> pd.Series:
    ### START CODE HERE ### (≈ 2 dòng)
    return pd.to_datetime(series)
    ### END CODE HERE ###


def parse_amount(series: pd.Series) -> pd.Series:
    ### START CODE HERE ### (≈ 1 dòng)
    return series.astype(float)
    ### END CODE HERE ###


print(parse_dates(pd.Series(["2024-03-05", "05/03/2024", "2024-12-31", "31/12/2024", "không rõ"])).tolist())
print(parse_amount(pd.Series(["120000000", "120.000.000", "5.000.000"])).tolist())

In [ ]:
check_equal(list(parse_dates(pd.Series(["2024-03-05", "05/03/2024"]))), [pd.Timestamp("2024-03-05")] * 2, "hai định dạng cùng một ngày (dd/mm, không phải mm/dd)")
check_true(parse_dates(pd.Series(["không rõ"])).isna().all(), "chuỗi lạ → NaT")
check_equal(list(parse_amount(pd.Series(["120000000", "120.000.000"]))), [120e6, 120e6], "hai định dạng số tiền")
truth4 = make_credit_applications(12000, seed=42).set_index("application_id")
sample4 = raw.drop_duplicates("application_id").set_index("application_id")
check_true((parse_dates(sample4.application_date).values == truth4.loc[sample4.index, "application_date"].values).all(), "khớp 100% ngày gốc trên toàn bộ dữ liệu thô")
check_true((parse_amount(sample4.loan_amount_vnd).values == truth4.loc[sample4.index, "loan_amount_vnd"].values).all(), "khớp 100% khoản vay gốc")

**Bài 5.** Viết `decide(p, loan, term_months, margin=0.05, lgd=0.70)` → mảng bool "duyệt" theo lợi nhuận kỳ vọng dương, và `profit(defaulted, approve, loan, term_months, margin, lgd)` → tổng lợi nhuận thực tế (đồng). Áp dụng trên hồ sơ 2023-H2 với mô hình học trên hồ sơ trước 07/2023.

In [ ]:
def decide(p, loan, term_months, margin=0.05, lgd=0.70):
    ### START CODE HERE ### (≈ 3 dòng)
    return np.asarray(p) < 0.5
    ### END CODE HERE ###


def profit(defaulted, approve, loan, term_months, margin=0.05, lgd=0.70):
    ### START CODE HERE ### (≈ 3 dòng)
    return 0.0
    ### END CODE HERE ###


train5 = apps[apps.application_date < "2023-07-01"]
val5 = apps[(apps.application_date >= "2023-07-01") & (apps.application_date < "2024-01-01")]
p5 = make_model().fit(train5, train5.defaulted).predict_proba(val5)[:, 1]
approve5 = decide(p5, val5.loan_amount_vnd, val5.term_months)
rule5 = ((val5.late_payments_12m == 0) & (val5.dti < 0.45)).values
print(f"duyệt {approve5.mean():.1%} | lợi nhuận mô hình {profit(val5.defaulted, approve5, val5.loan_amount_vnd, val5.term_months) / 1e9:.2f} tỉ | "
      f"luật {profit(val5.defaulted, rule5, val5.loan_amount_vnd, val5.term_months) / 1e9:.2f} tỉ")

In [ ]:
check_equal(list(decide([0.05, 0.10, 0.20], [100e6] * 3, [24] * 3)), [True, True, False], "ngưỡng 24 tháng = 12.5%")
check_equal(list(decide([0.10, 0.10], [100e6] * 2, [12, 36])), [False, True], "ngưỡng phụ thuộc kỳ hạn")
check_close(profit([0, 1, 0], [True, True, False], [100e6] * 3, [12] * 3), 5e6 - 70e6, message="lợi nhuận: +5tr, −70tr, 0")
check_greater(profit(val5.defaulted, approve5, val5.loan_amount_vnd, val5.term_months),
              profit(val5.defaulted, rule5, val5.loan_amount_vnd, val5.term_months), "trên validation, mô hình lãi hơn luật")

## L4 — Debug

**Bài 6.** Một học viên báo AUC 0.83 với code dưới. Có **ba** lỗi làm con số này vô nghĩa. Sửa trong hàm `honest_evaluation()` (giữ hồi quy logistic, dùng `make_model`) để trả về AUC trung thực trên hồ sơ 2024-H1 khi học trên hồ sơ trước 2024.

In [ ]:
def buggy_evaluation():
    data = apps.copy()
    data["defaulted"] = data.defaulted.fillna(0)                                         # lỗi?
    tr, te = train_test_split(data, test_size=0.2, random_state=0)                       # lỗi?
    model = make_model(NUMERIC + ["early_dpd30_3m"])                                     # lỗi?
    tr, te = tr.assign(early_dpd30_3m=tr.early_dpd30_3m.fillna(0)), te.assign(early_dpd30_3m=te.early_dpd30_3m.fillna(0))
    return roc_auc_score(te.defaulted, model.fit(tr, tr.defaulted).predict_proba(te)[:, 1])


def honest_evaluation():
    ### START CODE HERE ### (≈ 4 dòng)
    return buggy_evaluation()
    ### END CODE HERE ###


print(f"buggy: {buggy_evaluation():.3f} | honest: {honest_evaluation():.3f}")

In [ ]:
check_greater(buggy_evaluation(), 0.8, "tái hiện con số ảo")
check_less(honest_evaluation(), 0.78, "AUC trung thực thấp hơn nhiều")
check_greater(honest_evaluation(), 0.70, "nhưng mô hình vẫn có ích")

## L5 — Phân tích

**Bài 7.** **Stress test chính sách.** Mô hình học trên hồ sơ trước 2024 (có nhãn). Trên hồ sơ 2024-H1, nhân odds của xác suất dự đoán với $k \in \{1, 1.25, 1.5, 1.75, 2, 2.5, 3\}$ rồi áp dụng `decide`; lưu `stress` (DataFrame index $k$, cột `approval_rate`, `profit_bn` (tỉ đồng)). Đồng thời tính lợi nhuận của luật hiện tại `rule_profit_bn`. Nhận xét: chính sách tốt nhất nhạy thế nào với $k$?

In [ ]:
labeled7 = apps[apps.defaulted.notna()]
dev7 = labeled7[labeled7.application_date < "2024-01-01"]
test7 = labeled7[(labeled7.application_date >= "2024-01-01") & (labeled7.application_date < "2024-07-01")]
p7 = make_model().fit(dev7, dev7.defaulted).predict_proba(test7)[:, 1]
### START CODE HERE ### (≈ 7 dòng)
stress, rule_profit_bn = None, None
### END CODE HERE ###
print(stress.round(3), f"\nluật hiện tại: {rule_profit_bn:.2f} tỉ")

In [ ]:
check_equal(list(stress.index), [1, 1.25, 1.5, 1.75, 2, 2.5, 3], "đủ các mức k")
check_true(stress.approval_rate.is_monotonic_decreasing, "k tăng → duyệt ít hơn")
check_less(stress.loc[1, "profit_bn"], rule_profit_bn, "không điều chỉnh: thua luật cũ")
check_greater(stress.loc[2, "profit_bn"], rule_profit_bn, "điều chỉnh ×2: vượt luật cũ")
check_greater(stress.profit_bn.idxmax(), 1.4, "k tối ưu lớn hơn 1 rõ rệt")

## L6 — Thực tế

**Bài 8.** Viết `calibration_by_group(df, p, group_col, tolerance=0.25)` → DataFrame (index = nhóm) với cột `n`, `predicted` (TB xác suất), `actual` (tỉ lệ thật), `ratio` (= actual / predicted), `flag` (True nếu |ratio − 1| > tolerance). Áp dụng cho test 2024-H1 theo `employment_type` và theo `region`.

In [ ]:
def calibration_by_group(df, p, group_col, tolerance=0.25):
    ### START CODE HERE ### (≈ 5 dòng)
    return pd.DataFrame()
    ### END CODE HERE ###


by_emp = calibration_by_group(test7, p7, "employment_type")
by_region = calibration_by_group(test7, p7, "region")
print(by_emp.round(3), "\n")
print(by_region.round(3))

In [ ]:
check_equal(list(by_emp.columns), ["n", "predicted", "actual", "ratio", "flag"], "các cột")
check_equal(int(by_emp.n.sum()), len(test7), "mọi hồ sơ được tính")
check_close(by_emp.loc["salaried", "actual"], test7.defaulted[test7.employment_type == "salaried"].mean(), message="tỉ lệ thật nhóm salaried")
check_greater(by_emp.loc["self_employed", "ratio"], by_emp.loc["salaried", "ratio"], "cú sốc nặng hơn với lao động tự do")
check_true(by_region.flag.all(), "trong cú sốc, mọi vùng đều lệch calibration")

## L7 — Nâng cao

**Bài 9.** **Backtest có tính độ trễ của nhãn.** Mô phỏng triển khai thật: ở đầu mỗi quý $Q$ từ 2023-Q2 đến 2024-Q2, huấn luyện lại hồi quy logistic trên **những hồ sơ đã có nhãn tại thời điểm đó** (ngày nộp + 12 tháng ≤ ngày đầu quý), rồi đánh giá trên hồ sơ nộp trong quý $Q$. Lưu `backtest` (DataFrame index = quý dạng chuỗi `"2023Q2"`…, cột `n_train`, `auc`, `predicted`, `actual`).

In [ ]:
### START CODE HERE ### (≈ 10 dòng)
backtest = None
### END CODE HERE ###
print(backtest.round(3))

In [ ]:
check_equal(list(backtest.index), ["2023Q2", "2023Q3", "2023Q4", "2024Q1", "2024Q2"], "5 quý")
check_true(backtest.n_train.is_monotonic_increasing, "dữ liệu huấn luyện tăng dần theo thời gian")
check_less(backtest.loc["2023Q2", "n_train"], 1200, "đầu quý 2/2023 chỉ có nhãn của hồ sơ quý 1/2022")
check_less((backtest.loc[["2023Q2", "2023Q3", "2023Q4"], "actual"] - backtest.loc[["2023Q2", "2023Q3", "2023Q4"], "predicted"]).abs().max(), 0.025, "trước cú sốc: calibration tốt")
check_greater((backtest.loc[["2024Q1", "2024Q2"], "actual"] - backtest.loc[["2024Q1", "2024Q2"], "predicted"]).min(), 0.04, "sau cú sốc: đánh giá thấp rủi ro")
check_greater(backtest.loc["2024Q2", "auc"], 0.68, "khả năng xếp hạng vẫn giữ")